# RQ1 Accuracy and Efficiency

**Research question:** How does the trade-off between classification accuracy and computational cost affect sustainable waste classification?

This Kaggle notebook takes raw image-folder data as input and saves publication-ready tables as CSV and figures as PDF under `/kaggle/working/results/`.

In [1]:

# =========================
# COMMON SETUP
# =========================
import os, time, json, math, random, glob, shutil, pathlib, warnings
warnings.filterwarnings('ignore')
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import tensorflow as tf
from tensorflow.keras import layers, models
from tensorflow.keras.preprocessing import image_dataset_from_directory
from sklearn.metrics import classification_report, confusion_matrix, precision_recall_fscore_support, accuracy_score
from sklearn.model_selection import train_test_split

SEED = 42
random.seed(SEED); np.random.seed(SEED); tf.random.set_seed(SEED)

OUTPUT_DIR = Path('/kaggle/working/results')
FIG_DIR = OUTPUT_DIR / 'figures_pdf'
TAB_DIR = OUTPUT_DIR / 'tables_csv'
MODEL_DIR = OUTPUT_DIR / 'models'
for d in [FIG_DIR, TAB_DIR, MODEL_DIR]: d.mkdir(parents=True, exist_ok=True)

IMG_SIZE = (160, 160)
BATCH_SIZE = 32
EPOCHS = 3          # Increase to 10-20 for final results
MAX_IMAGES_PER_CLASS = 500  # Set None for full dataset; keep small for quick Kaggle runs
VALID_EXT = ('.jpg','.jpeg','.png','.webp','.bmp')

# Auto-detect dataset directory. Works with Kaggle datasets and uploaded zip-derived folders.
def find_image_root(base='/kaggle/input'):
    candidates=[]
    for root, dirs, files in os.walk(base):
        image_count=sum(1 for f in files if f.lower().endswith(VALID_EXT))
        if image_count>0:
            candidates.append((root, image_count))
    if not candidates:
        raise FileNotFoundError('No image files found under /kaggle/input. Please attach the dataset in Kaggle Notebook > Add Input.')
    # Prefer a root with class subfolders containing images; otherwise parent of deepest image dirs.
    best=max(candidates, key=lambda x: x[1])[0]
    # If best is a class folder, use parent when multiple sibling class folders exist.
    parent=str(Path(best).parent)
    sibling_img_dirs=[]
    for d in os.listdir(parent):
        p=os.path.join(parent,d)
        if os.path.isdir(p):
            n=sum(1 for f in os.listdir(p) if f.lower().endswith(VALID_EXT))
            if n>0: sibling_img_dirs.append(d)
    if len(sibling_img_dirs)>=2:
        return parent
    # Special case DATASET/TRAIN/TEST: use TRAIN as train root when present.
    for root, dirs, files in os.walk(base):
        if 'TRAIN' in dirs:
            return os.path.join(root,'TRAIN')
    return parent

DATA_ROOT = find_image_root('/kaggle/input')
print('Detected image root:', DATA_ROOT)
print('Class folders:', [d for d in os.listdir(DATA_ROOT) if os.path.isdir(os.path.join(DATA_ROOT,d))][:30])

def build_manifest(data_root=DATA_ROOT, max_per_class=MAX_IMAGES_PER_CLASS):
    rows=[]
    for cls in sorted(os.listdir(data_root)):
        cpath=os.path.join(data_root, cls)
        if not os.path.isdir(cpath): continue
        files=[]
        for r,_,fs in os.walk(cpath):
            files += [os.path.join(r,f) for f in fs if f.lower().endswith(VALID_EXT)]
        if not files: continue
        if max_per_class is not None and len(files)>max_per_class:
            files=random.sample(files, max_per_class)
        for f in files:
            rows.append({'image_path':f, 'label':cls})
    df=pd.DataFrame(rows)
    if df.empty: raise ValueError('No labeled images found. Expected class folders containing images.')
    return df

manifest = build_manifest()
manifest.to_csv(TAB_DIR/'dataset_manifest.csv', index=False)
print(manifest['label'].value_counts())
classes = sorted(manifest['label'].unique())
num_classes=len(classes)
label_to_idx={c:i for i,c in enumerate(classes)}

train_df, temp_df = train_test_split(manifest, test_size=0.30, stratify=manifest['label'], random_state=SEED)
val_df, test_df = train_test_split(temp_df, test_size=0.50, stratify=temp_df['label'], random_state=SEED)
for name,df in [('train',train_df),('val',val_df),('test',test_df)]:
    df.to_csv(TAB_DIR/f'{name}_split.csv', index=False)
    print(name, df.shape)

def make_ds(df, shuffle=False, augment=False):
    paths=df['image_path'].values
    labels=np.array([label_to_idx[x] for x in df['label'].values], dtype=np.int32)
    ds=tf.data.Dataset.from_tensor_slices((paths, labels))
    def load_img(path,label):
        img=tf.io.read_file(path)
        img=tf.image.decode_image(img, channels=3, expand_animations=False)
        img=tf.image.resize(img, IMG_SIZE)
        img=tf.cast(img, tf.float32)/255.0
        if augment:
            img=tf.image.random_flip_left_right(img)
            img=tf.image.random_brightness(img, 0.10)
        return img,label
    ds=ds.map(load_img, num_parallel_calls=tf.data.AUTOTUNE)
    if shuffle: ds=ds.shuffle(1000, seed=SEED)
    return ds.batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)

train_ds=make_ds(train_df, shuffle=True, augment=True)
val_ds=make_ds(val_df)
test_ds=make_ds(test_df)

def build_model(model_name):
    inputs=layers.Input(shape=IMG_SIZE+(3,))
    if model_name=='MobileNetV2':
        base=tf.keras.applications.MobileNetV2(include_top=False, weights='imagenet', input_tensor=inputs)
    elif model_name=='EfficientNetB0':
        # inputs already scaled 0-1; EfficientNet preprocessing disabled by using rescaling-neutral setup is acceptable for benchmark simplicity
        base=tf.keras.applications.EfficientNetB0(include_top=False, weights='imagenet', input_tensor=inputs)
    elif model_name=='ResNet50':
        base=tf.keras.applications.ResNet50(include_top=False, weights='imagenet', input_tensor=inputs)
    else:
        raise ValueError(model_name)
    base.trainable=False
    x=layers.GlobalAveragePooling2D()(base.output)
    x=layers.Dropout(0.25)(x)
    outputs=layers.Dense(num_classes, activation='softmax')(x)
    model=models.Model(inputs, outputs, name=model_name)
    model.compile(optimizer=tf.keras.optimizers.Adam(1e-3), loss='sparse_categorical_crossentropy', metrics=['accuracy'])
    return model

def evaluate_model(model, ds, df):
    y_true=[]; y_pred=[]; probs=[]
    for xb,yb in ds:
        p=model.predict(xb, verbose=0)
        probs.append(p); y_true.extend(yb.numpy().tolist()); y_pred.extend(np.argmax(p,axis=1).tolist())
    y_true=np.array(y_true); y_pred=np.array(y_pred); probs=np.vstack(probs)
    report=classification_report(y_true, y_pred, target_names=classes, output_dict=True, zero_division=0)
    rep_df=pd.DataFrame(report).T
    cm=confusion_matrix(y_true, y_pred, labels=list(range(num_classes)))
    return rep_df, cm, y_true, y_pred, probs

def measure_latency(model, ds, n_batches=10):
    # Warm-up
    for xb,yb in ds.take(1): model.predict(xb, verbose=0)
    total_imgs=0; start=time.time()
    for i,(xb,yb) in enumerate(ds.take(n_batches)):
        model.predict(xb, verbose=0)
        total_imgs += xb.shape[0]
    elapsed=time.time()-start
    return (elapsed/total_imgs)*1000 if total_imgs else np.nan

def save_pdf(fig, filename):
    path=FIG_DIR/filename
    fig.savefig(path, format='pdf', bbox_inches='tight')
    plt.close(fig)
    print('Saved', path)


2026-06-04 06:29:35.346580: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1780554575.636476      57 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1780554575.720668      57 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1780554576.494155      57 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1780554576.494225      57 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1780554576.494228      57 computation_placer.cc:177] computation placer alr

Detected image root: /kaggle/input/datasets/techsash/waste-classification-data/DATASET/TRAIN
Class folders: ['R', 'O']
label
O    500
R    500
Name: count, dtype: int64
train (700, 2)
val (150, 2)
test (150, 2)


2026-06-04 06:31:59.454281: E external/local_xla/xla/stream_executor/cuda/cuda_platform.cc:51] failed call to cuInit: INTERNAL: CUDA error: Failed call to cuInit: UNKNOWN ERROR (303)


In [2]:

# RQ1: Accuracy vs computational cost
model_names=['MobileNetV2','EfficientNetB0','ResNet50']
rows=[]
for mn in model_names:
    print('\nTraining', mn)
    model=build_model(mn)
    t0=time.time()
    hist=model.fit(train_ds, validation_data=val_ds, epochs=EPOCHS, verbose=1)
    train_time=time.time()-t0
    rep, cm, y_true, y_pred, probs=evaluate_model(model, test_ds, test_df)
    latency=measure_latency(model, test_ds)
    model_path=MODEL_DIR/f'{mn}.keras'; model.save(model_path)
    size_mb=os.path.getsize(model_path)/(1024*1024)
    rows.append({'model':mn,'test_accuracy':accuracy_score(y_true,y_pred),'macro_f1':rep.loc['macro avg','f1-score'],
                 'weighted_f1':rep.loc['weighted avg','f1-score'],'training_time_sec':train_time,
                 'inference_latency_ms_per_image':latency,'model_size_mb':size_mb})
metrics=pd.DataFrame(rows).sort_values('test_accuracy', ascending=False)
metrics.to_csv(TAB_DIR/'RQ1_accuracy_efficiency_table.csv', index=False)

fig, ax1 = plt.subplots(figsize=(8,5))
x=np.arange(len(metrics)); width=0.35
ax1.bar(x-width/2, metrics['test_accuracy'], width, label='Accuracy')
ax1.bar(x+width/2, metrics['macro_f1'], width, label='Macro F1')
ax1.set_xticks(x); ax1.set_xticklabels(metrics['model'], rotation=20, ha='right')
ax1.set_ylim(0,1); ax1.set_ylabel('Classification Performance')
ax2=ax1.twinx(); ax2.plot(x, metrics['inference_latency_ms_per_image'], marker='o', label='Latency')
ax2.set_ylabel('Latency (ms/image)')
ax1.set_title('RQ1: Accuracy–Efficiency Trade-off Across CNN Models')
ax1.legend(loc='upper left'); ax2.legend(loc='upper right')
fig.tight_layout(); save_pdf(fig,'RQ1_accuracy_efficiency_tradeoff.pdf')
metrics



Training MobileNetV2
9406464/9406464 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step
Epoch 1/3
22/22 ━━━━━━━━━━━━━━━━━━━━ 17s 467ms/step - accuracy: 0.7000 - loss: 0.6390 - val_accuracy: 0.8933 - val_loss: 0.3321
Epoch 2/3
22/22 ━━━━━━━━━━━━━━━━━━━━ 9s 374ms/step - accuracy: 0.8600 - loss: 0.3371 - val_accuracy: 0.9200 - val_loss: 0.2517
Epoch 3/3
22/22 ━━━━━━━━━━━━━━━━━━━━ 9s 370ms/step - accuracy: 0.8957 - loss: 0.2550 - val_accuracy: 0.9200 - val_loss: 0.2301

Training EfficientNetB0
16705208/16705208 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step
Epoch 1/3
22/22 ━━━━━━━━━━━━━━━━━━━━ 30s 866ms/step - accuracy: 0.4800 - loss: 0.7151 - val_accuracy: 0.5000 - val_loss: 0.7067
Epoch 2/3
22/22 ━━━━━━━━━━━━━━━━━━━━ 16s 633ms/step - accuracy: 0.5029 - loss: 0.7010 - val_accuracy: 0.5000 - val_loss: 0.7025
Epoch 3/3
22/22 ━━━━━━━━━━━━━━━━━━━━ 14s 640ms/step - accuracy: 0.5086 - loss: 0.7042 - val_accuracy: 0.4933 - val_loss: 0.6949

Training ResNet50
94765736/94765736 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step
Epoch 1/3
22/22 ━

,model,test_accuracy,macro_f1,weighted_f1,training_time_sec,inference_latency_ms_per_image,model_size_mb
0,MobileNetV2,0.840000,0.840000,0.840000,34.169586,12.329707,9.198833
2,ResNet50,0.513333,0.372816,0.372816,121.394071,40.144477,90.650299
1,EfficientNetB0,0.500000,0.333333,0.333333,60.136283,19.458302,16.278826
